# CEX Basic Usage

This notebook demonstrates the core CEX functionality after preprocessing: load a dataset, inspect its cell-type metadata, select a connectivity matrix, summarize one neuron, and construct FlyWire viewer URLs. CEX uses reassigned `id` values for table and matrix indexing and preserves dataset `rid` values for external services such as Codex and Neuroglancer.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import cex
from cex import get_dataset
from cex.neurons import Neuron
import cex.preprocessing as dprep

DATASET_NAME = "flywire"
REPO_ROOT = Path(cex.__file__).resolve().parents[2]
DATA_ROOT = REPO_ROOT / "data" / DATASET_NAME


## Optional Preprocessed Data Download

Set `PREPROCESSED_DOWNLOAD_URL` to the Dropbox link for the FlyWire preprocessed ZIP when the local tables are absent. The archive was created by zipping the `preprocessed` folder; extraction removes that one enclosing folder and writes its contents directly to `<codexplorer>/data/flywire/preprocessed/`. The downloaded archive is deleted after successful extraction.

In [2]:
PREPROCESSED_DOWNLOAD_URL = "https://www.dropbox.com/scl/fi/jo5yfjy6z9v7854f0blc7/flywire_preprocessed.zip?rlkey=qy7m3td1w2pp4orq0g5v5xmuf&dl=1"
PREPROCESSED_ARCHIVE_NAME = "flywire_preprocessed.zip"
DOWNLOAD_OVERWRITE_Q = False
PREPROCESSED_ROOT = DATA_ROOT / "preprocessed"

if PREPROCESSED_DOWNLOAD_URL is not None:
    print(f"Downloading and extracting preprocessed data from {PREPROCESSED_DOWNLOAD_URL} to {PREPROCESSED_ROOT}")
    dprep.download_and_extract(
        PREPROCESSED_DOWNLOAD_URL,
        PREPROCESSED_ROOT,
        PREPROCESSED_ARCHIVE_NAME,
        DOWNLOAD_OVERWRITE_Q,
        strip_single_root_Q=True,
    )

## Load the Dataset

Load the normalized tables after either downloading them above or generating them with preprocessing notebook `00`.

In [3]:
dataset = get_dataset(DATASET_NAME, DATA_ROOT)

## Cell Types

`cell_type_info` has one row per annotated type and records the order used by type-level connectivity matrices. Cell-level tables use assigned `id` values internally and retain source `rid` values for external identifiers.


In [4]:
dataset.cell.cell_type_info.head()

,type,num_cells,nt,flow,super_class,class,sub_class,hemilineage,nerve,group
0,4A0,2,GLUT,intrinsic,central,CX,tangential,putative_primary,NAN,FB
1,4A1,2,GLUT,intrinsic,central,CX,tangential,putative_primary,NAN,FB
2,4A10,4,DA,intrinsic,central,CX,tangential,DM6__prim,NAN,FB
3,4A2,16,GLUT,intrinsic,central,CX,tangential,DM6_dorso_medial,NAN,FB
4,4A21,4,GLUT,intrinsic,central,CX,tangential,DM6__prim,NAN,FB


In [5]:
# Single neuronal type IO statistics: 
dataset.show_cell_type_io_stat('Mi1', 'input', num_rows=10)
# Display the top 10 inputs of Mi1 neurons. 
# Cell type (Neurotransmitter, number of neurons in the dataset) direction
# First row: 
#   f_c: fraction of Mi1 neurons connected to L1 neurons.
#   n_c: average number of L1 that Mi1 is connected to 
#   en_c: effective number of L1 that Mi1 is connected to (count normalized by the connection with the maximum number of synapses). 
#   N_s: total number of synapses from L1 to Mi1 
#   F_s: Total fraction of synapses that Mi1 receives is from L1 
#   n_s: average number of synapses from L1 to Mi1
#   n_spc: average number of synapses Mi1 receives from each L1 
#   n_c_spc: for Mi1 that receive L1 input, the average number of synapses Mi1 receives from each L1.
#   NT: neurotransmitter of L1 neurons

Mi1 (ACH, # 1581) input (basic_pc)


,type,f_c,n_c,en_c,N_s,F_s,n_s,n_spc,n_c_spc,NT
0,L1,0.997,1.178,1.007,192286,0.289,121.623,111.756,112.111,GLUT
1,L5,0.994,3.886,1.150,82124,0.124,51.944,14.266,14.348,ACH
2,Pm08,1.000,4.443,2.769,54791,0.082,34.656,7.892,7.892,GABA
3,L3,0.905,1.245,0.971,26857,0.040,16.987,14.017,15.486,ACH
4,Pm01,0.998,4.496,2.341,26054,0.039,16.479,3.717,3.724,GABA
5,Mi13,0.970,3.438,1.861,25692,0.039,16.250,4.674,4.820,GABA
6,Dm1,0.946,1.939,1.402,24359,0.037,15.407,8.061,8.519,GLUT
7,OA-AL2b2,0.966,2.544,1.681,21989,0.033,13.908,5.498,5.692,ACH
8,Pm09,0.992,2.629,1.734,21108,0.032,13.351,5.385,5.426,GABA
9,C2,0.913,1.287,0.978,17332,0.026,10.963,8.802,9.637,GABA


## Connectivity Matrix

Select right-side L1 and Mi1 cells and return their directed cell-by-cell synapse-count matrix. The accompanying arrays identify every matrix row and column; `include_column_Q=True` also includes downloaded FlyWire `(p, q)` assignments when available.


In [6]:
matrix_data = dataset.connectivity.counts_between_types(pre_types=["L1", "Mi1"], 
                                                        post_types=["L5"],
                                                        pre_side="right", 
                                                        post_side="right",
                                                        include_column_Q=True)
matrix_data.keys()

dict_keys(['pre_types', 'post_types', 'pre_side', 'post_side', 'remove_autapse_Q', 'pre_ids', 'pre_rids', 'pre_cell_types', 'pre_type_to_idx', 'pre_type_neurotransmitter', 'pre_type_sign', 'pre_neurotransmitter', 'pre_sign', 'post_ids', 'post_rids', 'post_cell_types', 'post_type_to_idx', 'post_type_neurotransmitter', 'post_type_sign', 'post_neurotransmitter', 'post_sign', 'matrix', 'pre_column', 'post_column'])

In [7]:
# Number of synapses from [L1, Mi1] to L5 neurons in the right hemisphere. 
matrix_data['matrix']

<Compressed Sparse Row sparse array of dtype 'uint16'
	with 2178 stored elements and shape (1588, 785)>

## One Neuron

Construct one Mi1 neuron from its assigned `id`, then summarize its strongest input cell types. The same parsed connectivity identifies the five individual input and output partners with the most synapses and generates a Neuroglancer URL containing those cells. `open_url_Q=False` returns the URL without opening a browser.


In [8]:
cell_id = int(dataset.get_cell_type_id("Mi1", side="right")[0])
neuron = Neuron(cell_id, dataset, remove_autapse_Q=True)
print(f"Input statistics of neuron {cell_id} ({neuron.cell_type}):")
display(neuron.get_top_n_input_type_stat())

top_input_rids = sorted(neuron.input["rid2ns"], key=neuron.input["rid2ns"].get, reverse=True)[:5]
top_output_rids = sorted(neuron.output["rid2ns"], key=neuron.output["rid2ns"].get, reverse=True)[:5]
top_neighbor_ids = {
    "top 5 inputs": dataset.cell.rid_to_id_lookup.get_value(top_input_rids),
    "top 5 outputs": dataset.cell.rid_to_id_lookup.get_value(top_output_rids),
}
neuron.vis_self_w_ids_in_ng(top_neighbor_ids, open_url_Q=False)

Input statistics of neuron 59582 (Mi1):


,type,#s,#c,#ec,#s_w1c,#s/c,f_s,f_c
0,L1,157,1,1.00,157,157.00,0.3165,0.0130
1,L5,62,2,1.02,61,31.00,0.1250,0.0260
2,Pm08,45,5,2.37,19,9.00,0.0907,0.0649
3,L3,26,1,1.00,26,26.00,0.0524,0.0130
4,Dm1,24,3,1.41,17,8.00,0.0484,0.0390
5,Mi13,19,4,1.58,12,4.75,0.0383,0.0519
6,C2,18,1,1.00,18,18.00,0.0363,0.0130
7,Pm01,18,4,2.25,8,4.50,0.0363,0.0519
8,OA-AL2b2,14,3,1.27,11,4.67,0.0282,0.0390
9,R8,13,1,1.00,13,13.00,0.0262,0.0130


'https://spelunker.cave-explorer.org/#!%7B%22dimensions%22%3A%7B%22x%22%3A%5B4e-09%2C%22m%22%5D%2C%22y%22%3A%5B4e-09%2C%22m%22%5D%2C%22z%22%3A%5B4e-08%2C%22m%22%5D%7D%2C%22position%22%3A%5B130944.1484%2C56685.9532%2C4156.3672%5D%2C%22crossSectionScale%22%3A1%2C%22projectionScale%22%3A90000%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22source%22%3A%22precomputed%3A%2F%2Fhttps%3A%2F%2Fbossdb-open-data.s3.amazonaws.com%2Fflywire%2Ffafbv14%22%2C%22tab%22%3A%22source%22%2C%22name%22%3A%22EM%22%7D%2C%7B%22type%22%3A%22segmentation%22%2C%22source%22%3A%22precomputed%3A%2F%2Fgs%3A%2F%2Fflywire_neuropil_meshes%2Fwhole_neuropil%2Fbrain_mesh_v3%22%2C%22tab%22%3A%22source%22%2C%22objectAlpha%22%3A0.05%2C%22hideSegmentZero%22%3Afalse%2C%22segments%22%3A%5B%221%22%5D%2C%22segmentColors%22%3A%7B%221%22%3A%22%23b5b5b5%22%7D%2C%22name%22%3A%22brain_mesh_v3%22%7D%2C%7B%22type%22%3A%22segmentation%22%2C%22source%22%3A%22precomputed%3A%2F%2Fgs%3A%2F%2Fflywire_v141_m783%22%2C%22tab%22%3A%22segments%

## FlyWire Link

For a simpler view, convert one or more assigned IDs to FlyWire root IDs and return a Neuroglancer URL directly. URL construction remains side-effect free unless `open_url_Q=True` is requested explicitly.


In [9]:
dataset.ng_url_for_ids([cell_id])

'https://spelunker.cave-explorer.org/#!%7B%22dimensions%22%3A%7B%22x%22%3A%5B4e-09%2C%22m%22%5D%2C%22y%22%3A%5B4e-09%2C%22m%22%5D%2C%22z%22%3A%5B4e-08%2C%22m%22%5D%7D%2C%22position%22%3A%5B130944.1484%2C56685.9532%2C4156.3672%5D%2C%22crossSectionScale%22%3A1%2C%22projectionScale%22%3A90000%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22source%22%3A%22precomputed%3A%2F%2Fhttps%3A%2F%2Fbossdb-open-data.s3.amazonaws.com%2Fflywire%2Ffafbv14%22%2C%22tab%22%3A%22source%22%2C%22name%22%3A%22EM%22%7D%2C%7B%22type%22%3A%22segmentation%22%2C%22source%22%3A%22precomputed%3A%2F%2Fgs%3A%2F%2Fflywire_neuropil_meshes%2Fwhole_neuropil%2Fbrain_mesh_v3%22%2C%22tab%22%3A%22source%22%2C%22objectAlpha%22%3A0.05%2C%22hideSegmentZero%22%3Afalse%2C%22segments%22%3A%5B%221%22%5D%2C%22segmentColors%22%3A%7B%221%22%3A%22%23b5b5b5%22%7D%2C%22name%22%3A%22brain_mesh_v3%22%7D%2C%7B%22type%22%3A%22segmentation%22%2C%22source%22%3A%22precomputed%3A%2F%2Fgs%3A%2F%2Fflywire_v141_m783%22%2C%22tab%22%3A%22segments%